In [1]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 🛠️ CONFIGURAÇÕES
# ==============================================================================

# Caminho do arquivo que você acabou de gerar (O "FULL")
caminho_arquivo = r"\\cet-020059\DES_Dados\Dados\ARQUIVO DE CÓDIGOS\Funções Compartilhadas\DB_DES\data\cet\dados_cet_pre_tratados_20251216_ENRIQUECIDO_FULL.xlsx"

# 🗺️ REGRAS DE HIERARQUIA (Sua lista personalizada)
# Chave = Sigla da GET
# Valor = Lista de DETs permitidas
REGRAS_HIERARQUIA = {
    'CN': ['CN1', 'CN2', 'CN3'],          
    'LE': ['LE1', 'LE2', 'LE3', 'LE4'],    
    'SE': ['SE1', 'SE2', 'SE3'],           
    'SU': ['SU1', 'SU2', 'SU3', 'SU4'],    
    'SO': ['SO1', 'SO2', 'SO3'],           
    'OE': ['OE1', 'OE2', 'OE3', 'OE4'],    
    'NO': ['NO1', 'NO2', 'NO3'],           
    'MB': ['PB', 'TT', 'TS']               # Marginais
}

print("✅ Configurações carregadas.")

✅ Configurações carregadas.


In [2]:
print(f"📂 Carregando arquivo: {caminho_arquivo}...")

if caminho_arquivo.endswith('.csv'):
    df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8', dtype=str)
else:
    df = pd.read_excel(caminho_arquivo, dtype=str)

# Cria a referência da linha do Excel (Index + 2)
# (+1 pq começa em 0, +1 pelo cabeçalho)
df['LINHA_EXCEL'] = df.index + 2

print(f"📊 Total de linhas carregadas: {len(df)}")
display(df[['LINHA_EXCEL', 'GET', 'DET', 'latitude_geocode']].head())

📂 Carregando arquivo: \\cet-020059\DES_Dados\Dados\ARQUIVO DE CÓDIGOS\Funções Compartilhadas\DB_DES\data\cet\dados_cet_pre_tratados_20251216_ENRIQUECIDO_FULL.xlsx...
📊 Total de linhas carregadas: 201897


,LINHA_EXCEL,GET,DET,latitude_geocode
0,2,NO,NO3,-23.451666525
1,3,NO,NO3,-23.451666525
2,4,NO,NO3,-23.451666525
3,5,NO,NO3,-23.451666525
4,6,NO,NO3,-23.451666525


In [3]:
def auditar_hierarquia(row):
    """
    Verifica se a combinação GET x DET é válida segundo as regras da CET.
    """
    # 1. Limpeza dos dados (Remove 'GET', 'DET', espaços, etc)
    def limpar(txt):
        if pd.isna(txt): return ""
        # Remove prefixos comuns e sujeira
        return str(txt).upper()\
            .replace('GET', '')\
            .replace('DET', '')\
            .replace('SQLPRODUCAO:', '')\
            .strip()

    raw_get = row.get('GET')
    raw_det = row.get('DET')

    sigla_get = limpar(raw_get)
    sigla_det = limpar(raw_det)
    
    # 2. Se estiver vazio, ignoramos (não é erro de hierarquia, é falta de dado)
    if not sigla_get or not sigla_det:
        return "IGNORAR (Vazio)"
    
    # 3. Validação
    if sigla_get in REGRAS_HIERARQUIA:
        lista_permitida = REGRAS_HIERARQUIA[sigla_get]
        
        if sigla_det in lista_permitida:
            return "OK"
        else:
            return f"❌ ERRO: {sigla_get} não contém {sigla_det}"
            
    # Caso apareça uma GET que não está no seu dicionário (ex: 'G5')
    return f"⚠️ ALERTA: GET '{sigla_get}' desconhecida"

print("✅ Função de auditoria compilada.")

✅ Função de auditoria compilada.


In [4]:
print("🕵️‍♂️ Iniciando auditoria...")

# Aplica a validação
df['STATUS_AUDITORIA'] = df.apply(auditar_hierarquia, axis=1)

# Filtra apenas os problemas
erros = df[
    ~df['STATUS_AUDITORIA'].isin(['OK', 'IGNORAR (Vazio)'])
].copy()

# =========================================================
# 📊 RELATÓRIO FINAL
# =========================================================
total = len(df)
ok = len(df[df['STATUS_AUDITORIA'] == 'OK'])
vazios = len(df[df['STATUS_AUDITORIA'] == 'IGNORAR (Vazio)'])
com_erro = len(erros)

print("\n" + "="*60)
print("📋 RESUMO DA VALIDAÇÃO")
print("="*60)
print(f"Total Analisado: {total}")
print(f"✅ Hierarquia Correta: {ok} ({(ok/total)*100:.1f}%)")
print(f"⚪ Vazios/Incompletos: {vazios}")
print(f"❌ Inconsistências:    {com_erro}")
print("="*60)

if com_erro > 0:
    print("\n🔍 TOP 10 ERROS ENCONTRADOS (Vá nestas linhas no Excel):")
    cols_show = ['LINHA_EXCEL', 'GET', 'DET', 'STATUS_AUDITORIA', 'latitude_geocode', 'longitude_geocode']
    display(erros[cols_show].head(10))
    
    print("\n📈 CONTAGEM DOS TIPOS DE ERRO:")
    resumo_erros = erros['STATUS_AUDITORIA'].value_counts().reset_index()
    resumo_erros.columns = ['Tipo de Erro', 'Quantidade']
    display(resumo_erros)
else:
    print("\n🎉 PARABÉNS! Nenhuma inconsistência de hierarquia encontrada.")

🕵️‍♂️ Iniciando auditoria...

📋 RESUMO DA VALIDAÇÃO
Total Analisado: 201897
✅ Hierarquia Correta: 190076 (94.1%)
⚪ Vazios/Incompletos: 5999
❌ Inconsistências:    5822

🔍 TOP 10 ERROS ENCONTRADOS (Vá nestas linhas no Excel):


,LINHA_EXCEL,GET,DET,STATUS_AUDITORIA,latitude_geocode,longitude_geocode
99,101,MB,LE1,❌ ERRO: MB não contém LE1,-23.527052505,-46.59905052
202,204,OE,SO2,❌ ERRO: OE não contém SO2,-23.61017502315789,-46.72708316210526
234,236,SO,SU2,❌ ERRO: SO não contém SU2,-23.57257398,-46.64252772
238,240,SO,PB,❌ ERRO: SO não contém PB,-23.64462198,-46.72742184
248,250,CN,SU2,❌ ERRO: CN não contém SU2,-23.556253995,-46.63730052
272,274,OE,NO1,❌ ERRO: OE não contém NO1,-23.535748584,-46.71728625599999
312,314,MB,CN1,❌ ERRO: MB não contém CN1,-23.5152099864,-46.6589459376
352,354,CN,NO2,❌ ERRO: CN não contém NO2,-23.49084402,-46.67579387999999
361,363,MB,NO3,❌ ERRO: MB não contém NO3,-23.51692107,-46.73584462153846
379,381,MB,LE1,❌ ERRO: MB não contém LE1,-23.527052505,-46.59905052



📈 CONTAGEM DOS TIPOS DE ERRO:


,Tipo de Erro,Quantidade
0,❌ ERRO: MB não contém CN1,750
1,❌ ERRO: SO não contém SU2,693
2,❌ ERRO: SO não contém SU3,491
3,❌ ERRO: MB não contém CN2,482
4,❌ ERRO: OE não contém NO1,465
5,❌ ERRO: CN não contém NO2,407
6,❌ ERRO: MB não contém LE1,402
7,❌ ERRO: OE não contém SO2,393
8,❌ ERRO: OE não contém SO1,298
9,❌ ERRO: CN não contém LE1,237


In [5]:
if com_erro > 0:
    nome_relatorio = "RELATORIO_ERROS_HIERARQUIA.xlsx"
    print(f"💾 Salvando lista de erros em: {nome_relatorio}")
    erros.to_excel(nome_relatorio, index=False)
    print("✅ Salvo com sucesso.")
else:
    print("👍 Nada para salvar.")

💾 Salvando lista de erros em: RELATORIO_ERROS_HIERARQUIA.xlsx
✅ Salvo com sucesso.


In [6]:
import os
print("📂 O arquivo foi salvo aqui:")
print(os.getcwd())

📂 O arquivo foi salvo aqui:
c:\Users\joao.fernandez\Documents\Projeto_Padronizacao_Logradouros\PROJETO
